# Hedonic Features and Targets

Creates quality-adjusted price indices using hedonic regression, then builds features and forward-looking targets for model training.


## Setup


In [4]:
from pathlib import Path
import sys
import warnings

import re, shutil
import numpy as np
import pandas as pd

import statsmodels.api as sm
from patsy import dmatrices

PROJECT_ROOT = Path('..').resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

INTERIM = Path('../data/interim')
PROCESSED = Path('../data/processed')
INTERIM.mkdir(parents=True, exist_ok=True)
PROCESSED.mkdir(parents=True, exist_ok=True)

warnings.filterwarnings('ignore', category=FutureWarning)


## Load LA×Month Datasets


In [7]:
tx_path = INTERIM / 'ppd_la_month_transactions.parquet'
agg_path = INTERIM / 'la_month_prices.parquet'
assert tx_path.exists(), f'Missing {tx_path}'
assert agg_path.exists(), f'Missing {agg_path}'

tx = pd.read_parquet(tx_path)
agg = pd.read_parquet(agg_path)
print('Transactions shape:', tx.shape)
print('Aggregate shape:', agg.shape)
tx.head()


Transactions shape: (30218056, 10)
Aggregate shape: (117249, 7)


,date,month,LA_code,LA_name,price,log_price,postcode,property_type,old_new,duration
0,1995-07-07,1995-07-31,E06000042,Milton Keynes,70000,11.156251,MK159HP,D,N,F
1,1995-02-03,1995-02-28,E08000024,Sunderland,44500,10.703244,SR60AQ,T,N,F
2,1995-01-13,1995-01-31,E07000067,Braintree,56500,10.941996,CO61SQ,T,N,F
3,1995-07-28,1995-07-31,E08000029,Solihull,58000,10.968198,B904TG,T,N,F
4,1995-06-28,1995-06-30,E08000027,Dudley,51000,10.839581,DY51SA,S,N,F


## Hedonic Adjustment


In [10]:
mix_cols = ['price','LA_code','month','property_type','duration','old_new']
z = tx.dropna(subset=mix_cols).copy()

# sanitise categories
z['property_type'] = z['property_type'].astype(str).str.upper()
z['duration']      = z['duration'].astype(str).str.upper()
z['old_new']       = z['old_new'].astype(str).str.upper()

# log price (safe)
z['log_price'] = np.log(z['price'].clip(lower=1))

# bucket to reduce size
bucket = (z.groupby(['LA_code','month','property_type','duration','old_new'], as_index=False)
            .agg(log_price_med=('log_price','median'),
                 n_obs=('log_price','size')))

# patsy likes strings for month cats
bucket['month_cat'] = bucket['month'].dt.strftime('%Y-%m')

# build design matrices; use WLS with frequency weights
formula = 'log_price_med ~ C(LA_code) + C(month_cat) + C(property_type) + C(duration) + C(old_new)'
y, X = dmatrices(formula, data=bucket, return_type='dataframe')
w = bucket['n_obs'].values

model = sm.WLS(y, X, weights=w).fit()
print(model.summary().tables[1])  # quick sanity

# predict per bucket, then aggregate to LA×month
bucket['log_hat']   = model.predict(X)
bucket['price_hat'] = np.exp(bucket['log_hat'])
# (optional log-normal bias correction:)
# bucket['price_hat'] = np.exp(bucket['log_hat'] + 0.5 * model.mse_resid)

hedo = (bucket.groupby(['LA_code','month'], as_index=False)
              .agg(hedonic_price=('price_hat','median'),
                   n_obs=('n_obs','sum')))

# build base-100 index per LA, aligned row-for-row
hedo = hedo.sort_values(['LA_code','month']).reset_index(drop=True)
base = hedo.groupby('LA_code')['hedonic_price'].transform('first')
hedo['hedonic_ix'] = 100 * hedo['hedonic_price'] / base

hedo_path = INTERIM / 'la_month_hedonic.parquet'
hedo.to_parquet(hedo_path, index=False, compression='zstd')
print('Saved hedonic index →', hedo_path, f'({len(hedo):,} rows)')
hedo.head()


                              coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------
Intercept                  10.6467      0.006   1747.394      0.000      10.635      10.659
C(LA_code)[T.E06000002]     0.0591      0.006     10.422      0.000       0.048       0.070
C(LA_code)[T.E06000003]     0.1632      0.006     28.780      0.000       0.152       0.174
C(LA_code)[T.E06000004]     0.1970      0.005     37.666      0.000       0.187       0.207
C(LA_code)[T.E06000005]     0.2088      0.006     36.345      0.000       0.198       0.220
C(LA_code)[T.E06000006]     0.2786      0.006     46.788      0.000       0.267       0.290
C(LA_code)[T.E06000007]     0.5506      0.005    105.652      0.000       0.540       0.561
C(LA_code)[T.E06000008]     0.1484      0.006     26.496      0.000       0.137       0.159
C(LA_code)[T.E06000009]     0.2178      0.005     40.568      0.000       0.207 

,LA_code,month,hedonic_price,n_obs,hedonic_ix
0,E06000001,1995-01-31,26172.301189,86,100.000000
1,E06000001,1995-02-28,26026.923857,86,99.444537
2,E06000001,1995-03-31,26080.580175,117,99.649549
3,E06000001,1995-04-30,26943.653925,76,102.947210
4,E06000001,1995-05-31,26342.860263,100,100.651678


## Merge Hedonic with Median Panel


In [13]:
# build the base LA×month panel with full history

keys = ['LA_code', 'month']

# Outer-join so i KEEP months that exist in either source (pre-2015 hedonic-only months included)
panel = pd.merge(hedo, agg, on=keys, how='outer', suffixes=('', '_agg'))

if 'n_sales' not in panel.columns and 'n' in panel.columns:
    panel = panel.rename(columns={'n': 'n_sales'})

if 'n' in panel.columns:
    panel['n_sales'] = panel['n_sales'].fillna(panel['n'])
    panel.drop(columns=['n'], inplace=True)

# Choose the price series used for features/targets:
# prefer hedonic_price when present, else fall back to median_price
panel['px_used'] = panel['hedonic_price'].fillna(panel['median_price'])

# Sort and sanity check the date span
panel = panel.sort_values(['LA_code', 'month']).reset_index(drop=True)

print(
    "[PANEL SPAN] ",
    panel['month'].min().date(), "→", panel['month'].max().date(),
    "| LAs:", panel['LA_code'].nunique(),
    "| months uniq:", panel['month'].nunique()
)
display(panel.head())


[PANEL SPAN]  1995-01-31 → 2025-09-30 | LAs: 318 | months uniq: 369


,LA_code,month,hedonic_price,n_obs,hedonic_ix,LA_name,median_price,mean_price,n_sales,is_thin,px_used
0,E06000001,1995-01-31,26172.301189,86,100.000000,Hartlepool,32750.0,40276.104651,86,False,26172.301189
1,E06000001,1995-02-28,26026.923857,86,99.444537,Hartlepool,35725.0,40713.488372,86,False,26026.923857
2,E06000001,1995-03-31,26080.580175,117,99.649549,Hartlepool,32500.0,39692.393162,117,False,26080.580175
3,E06000001,1995-04-30,26943.653925,76,102.947210,Hartlepool,32000.0,39239.618421,76,False,26943.653925
4,E06000001,1995-05-31,26342.860263,100,100.651678,Hartlepool,36975.0,41924.500000,100,False,26342.860263


## Attach Context Features


In [16]:
#  Slow sources to merge into the panel
slow_sources = [
    # file,                     columns to keep
    ('broadband_la.parquet',    ['LA_code', 'gigabit_pct']),
    ('households_la.parquet',   ['LA_code', 'households_2021']),
    ('rent_index.parquet',      ['LA_code', 'month', 'rent_index']),
]

def _coalesce(panel_df: pd.DataFrame, base_col: str, new_col: str, is_pct: bool = False, tol: float | None = None):
    """
    Prefer source values (new_col) when present; keep base when new is NaN or
    when both are (near-)identical within tol. Enforce sensible bounds for % fields.
    """
    b = panel_df[base_col] if base_col in panel_df.columns else np.nan
    n = panel_df[new_col]
    if tol is None:
        tol = 1e-6 if not is_pct else 1e-3

    out = b.copy() if base_col in panel_df.columns else pd.Series(np.nan, index=panel_df.index)

    if is_pct:
        valid_b = (out.between(0, 100)) | out.isna()
        valid_n = (n.between(0, 100)) | n.isna()
    else:
        valid_b = (out >= 0) | out.isna()
        valid_n = (n >= 0) | n.isna()

    take_new   = out.isna() & n.notna()
    both       = out.notna() & n.notna()
    same       = (out - n).abs() <= tol
    different  = both & (~same)
    prefer_new = different & ((~valid_b & valid_n) | (valid_b & valid_n))

    final = out.copy()
    idx = take_new | prefer_new
    final.loc[idx] = n.loc[idx]

    panel_df[base_col] = final
    panel_df.drop(columns=[new_col], inplace=True, errors='ignore')
    return panel_df

# Merge each slow source
for fname, cols in slow_sources:
    path = INTERIM / fname
    if not path.exists():
        print(f"[WARN] Skipping {fname} (not found).")
        continue

    df   = pd.read_parquet(path)[cols].copy()
    keys = ['LA_code', 'month'] if 'month' in cols else ['LA_code']

    # Merge with suffixes so we can coalesce duplicates deterministically
    panel = panel.merge(df, on=keys, how='left', suffixes=('', '_new'))

    # Coalesce any duplicate-measure columns that appeared with _new
    for c in cols:
        if c in keys:
            continue
        new_name = f'{c}_new'
        if new_name in panel.columns:
            panel = _coalesce(panel, c, new_name, is_pct=(c == 'gigabit_pct'))

# Drop any leftover *_new / *_dup columns
leftovers = [c for c in panel.columns if c.endswith('_new') or c.endswith('_dup')]
if leftovers:
    panel.drop(columns=leftovers, inplace=True, errors='ignore')

# CTX: merge NB4 context files
ctx_static_path = INTERIM / "context/context_static.parquet"   # LA-level (no time)
ctx_time_path   = INTERIM / "context/context_time.parquet"     # LA×month

def _first_non_null(a: pd.Series | None, b: pd.Series | None):
    if a is None and b is None: return None
    if a is None: return b
    if b is None: return a
    return a.where(a.notna(), b)

def _collapse_xy_suffixes(df: pd.DataFrame) -> pd.DataFrame:
    """
    Generic cleanup for any lingering *_x / *_y columns from previous runs.
    If base exists, fill it from *_x/*_y where base is NaN; otherwise rename *_x/*_y to base.
    """
    to_drop = []
    for col in list(df.columns):
        if col.endswith(('_x', '_y')):
            base = col[:-2]
            if base in df.columns:
                df[base] = _first_non_null(df[base], df[col])
                to_drop.append(col)
            else:
                df.rename(columns={col: base}, inplace=True)
    if to_drop:
        df.drop(columns=to_drop, inplace=True, errors="ignore")
    return df

# Clean up any old *_x / *_y before new context merges
panel = _collapse_xy_suffixes(panel)

# Static context: merge on LA_code; avoid clobbering LA_name
if ctx_static_path.exists():
    ctxs = pd.read_parquet(ctx_static_path)

    # Do not bring in a second LA_name
    if "LA_name" in ctxs.columns:
        ctxs = ctxs.drop(columns=["LA_name"])

    # Only keep columns that are not ALL-NaN
    ctxs = ctxs.loc[:, ctxs.notna().any(axis=0)]

    before_cols = set(panel.columns)
    panel = panel.merge(
        ctxs,
        on="LA_code",
        how="left",
        validate="many_to_one",
        suffixes=("", "_ctx"),
    )
    added = sorted(list(set(panel.columns) - before_cols))
    print(f"[CTX] Added STATIC cols ({len(added)}): {added[:10]}{' …' if len(added)>10 else ''}")

    # Coalesce any known overlaps, preferring panel values
    for base in ["households_2021", "gigabit_pct", "area_km2",
                 "fastfood_per_1k_hh", "supermkt_per_1k_hh",
                 "stops_per_km2", "stops_per_10k_hh",
                 "imd_score_mean", "airbnb_per_10k_hh"]:
        b, c = base, f"{base}_ctx"
        if b in panel.columns and c in panel.columns:
            panel[b] = _first_non_null(panel[b], panel[c])
            panel.drop(columns=[c], inplace=True, errors="ignore")

    # Coalesce z-scored variants if present
    for base_z in [f"{x}_z" for x in [
        "fastfood_per_1k_hh", "supermkt_per_1k_hh",
        "stops_per_km2", "stops_per_10k_hh",
        "imd_score_mean", "airbnb_per_10k_hh"
    ]]:
        b, c = base_z, f"{base_z}_ctx"
        if b in panel.columns and c in panel.columns:
            panel[b] = _first_non_null(panel[b], panel[c])
            panel.drop(columns=[c], inplace=True, errors="ignore")
else:
    print("[CTX] context_static.parquet not found — skip.")

# Time context: merge on LA_code + month
if ctx_time_path.exists():
    ctxt = pd.read_parquet(ctx_time_path)
    # Only keep useful columns (not all-NaN)
    ctxt = ctxt.loc[:, ctxt.notna().any(axis=0)]

    before_cols = set(panel.columns)
    panel = panel.merge(
        ctxt,
        on=["LA_code","month"],
        how="left",
        validate="many_to_one",
        suffixes=("", "_ctx"),
    )
    added = sorted(list(set(panel.columns) - before_cols))
    print(f"[CTX] Added TIME cols ({len(added)}): {added[:10]}{' …' if len(added)>10 else ''}")

    # Coalesce time-series overlaps we know about
    for base in ["iphrp_england", "iphrp_yoy", "iphrp_england_z", "iphrp_yoy_z"]:
        b, c = base, f"{base}_ctx"
        if b in panel.columns and c in panel.columns:
            panel[b] = _first_non_null(panel[b], panel[c])
            panel.drop(columns=[c], inplace=True, errors="ignore")
else:
    print("[CTX] context_time.parquet not found — skip.")

# Coverage diagnostics (BEFORE FILTERING)
def _pct_non_null(s: pd.Series) -> float:
    return (1.0 - s.isna().mean()) * 100.0

cov_gig = _pct_non_null(panel.get('gigabit_pct',     pd.Series(index=panel.index)))
cov_hh  = _pct_non_null(panel.get('households_2021', pd.Series(index=panel.index)))
cov_rnt = _pct_non_null(panel.get('rent_index',      pd.Series(index=panel.index)))

print(f"[COVERAGE] gigabit_pct: {cov_gig:.1f}% | households_2021: {cov_hh:.1f}% | rent_index: {cov_rnt:.1f}%")

ri_path = INTERIM / 'rent_index.parquet'
if ri_path.exists():
    ri = pd.read_parquet(ri_path)
    if not ri.empty:
        ri_min, ri_max = ri['month'].min(), ri['month'].max()
        p_min,  p_max  = panel['month'].min(), panel['month'].max()
        print(f"[DIAG] rent_index.parquet months: {ri_min.date()} → {ri_max.date()} ({ri['month'].nunique()} months)")
        print(f"[DIAG] panel months:           {p_min.date()} → {p_max.date()} ({panel['month'].nunique()} months)")

        pre_mask  = panel['month'] <  pd.Timestamp('2015-01-31')
        post_mask = panel['month'] >= pd.Timestamp('2015-01-31')
        cov_pre   = _pct_non_null(panel.loc[pre_mask,  'rent_index']) if pre_mask.any()  else float('nan')
        cov_post  = _pct_non_null(panel.loc[post_mask, 'rent_index']) if post_mask.any() else float('nan')
        print(f"[COVERAGE by period] pre-2015: {cov_pre:.1f}% | 2015+ : {cov_post:.1f}%")

display(panel.head())

# Persist BEFORE-FILTER version (ALL history for CORE)
OUT_ALL = INTERIM / "la_month_panel_ALL.parquet"
panel.to_parquet(OUT_ALL, index=False)
print(f"\nSaved merged panel (ALL history) → {OUT_ALL} "
      f"(rows={len(panel):,}, LAs={panel['LA_code'].nunique()}, months={panel['month'].nunique()})")

# Build AFTER-FILTER version (2015+ for WITHRENT)
panel_2015p = panel.loc[panel['month'] >= '2015-01-31'].copy()

cov_gig_af = _pct_non_null(panel_2015p.get('gigabit_pct',     pd.Series(index=panel_2015p.index)))
cov_hh_af  = _pct_non_null(panel_2015p.get('households_2021', pd.Series(index=panel_2015p.index)))
cov_rnt_af = _pct_non_null(panel_2015p.get('rent_index',      pd.Series(index=panel_2015p.index)))
print(f"[AFTER FILTER 2015+] gigabit_pct: {cov_gig_af:.1f}% | households_2021: {cov_hh_af:.1f}% | rent_index: {cov_rnt_af:.1f}%")
display(panel_2015p.head())

OUT_2015P = INTERIM / "la_month_panel_2015plus.parquet"
panel_2015p.to_parquet(OUT_2015P, index=False)
print(f"Saved merged panel (2015+) → {OUT_2015P} "
      f"(rows={len(panel_2015p):,}, LAs={panel_2015p['LA_code'].nunique()}, months={panel_2015p['month'].nunique()})")


# Cell 5 should explicitly load one of:
#   - CORE features:     ../data/interim/la_month_panel_ALL.parquet
#   - WITHRENT features: ../data/interim/la_month_panel_2015plus.parquet

[CTX] Added STATIC cols (14): ['airbnb_per_10k_hh', 'airbnb_per_10k_hh_z', 'area_km2', 'fastfood_per_1k_hh', 'fastfood_per_1k_hh_z', 'households_2021_ctx', 'imd_score_mean', 'imd_score_mean_z', 'stops_per_10k_hh', 'stops_per_10k_hh_z'] …
[CTX] Added TIME cols (4): ['iphrp_england', 'iphrp_england_z', 'iphrp_yoy', 'iphrp_yoy_z']
[COVERAGE] gigabit_pct: 99.4% | households_2021: 99.4% | rent_index: 31.7%
[DIAG] rent_index.parquet months: 2015-01-31 → 2025-09-30 (129 months)
[DIAG] panel months:           1995-01-31 → 2025-09-30 (369 months)
[COVERAGE by period] pre-2015: 0.0% | 2015+ : 90.6%


,LA_code,month,hedonic_price,n_obs,hedonic_ix,LA_name,median_price,mean_price,n_sales,is_thin,...,fastfood_per_1k_hh_z,supermkt_per_1k_hh_z,stops_per_km2_z,stops_per_10k_hh_z,imd_score_mean_z,airbnb_per_10k_hh_z,iphrp_england,iphrp_yoy,iphrp_england_z,iphrp_yoy_z
0,E06000001,1995-01-31,26172.301189,86,100.000000,Hartlepool,32750.0,40276.104651,86,False,...,0.410539,-0.489004,0.084288,0.476515,1.949175,NaN,NaN,NaN,NaN,NaN
1,E06000001,1995-02-28,26026.923857,86,99.444537,Hartlepool,35725.0,40713.488372,86,False,...,0.410539,-0.489004,0.084288,0.476515,1.949175,NaN,NaN,NaN,NaN,NaN
2,E06000001,1995-03-31,26080.580175,117,99.649549,Hartlepool,32500.0,39692.393162,117,False,...,0.410539,-0.489004,0.084288,0.476515,1.949175,NaN,NaN,NaN,NaN,NaN
3,E06000001,1995-04-30,26943.653925,76,102.947210,Hartlepool,32000.0,39239.618421,76,False,...,0.410539,-0.489004,0.084288,0.476515,1.949175,NaN,NaN,NaN,NaN,NaN
4,E06000001,1995-05-31,26342.860263,100,100.651678,Hartlepool,36975.0,41924.500000,100,False,...,0.410539,-0.489004,0.084288,0.476515,1.949175,NaN,NaN,NaN,NaN,NaN



Saved merged panel (ALL history) → ../data/interim/la_month_panel_ALL.parquet (rows=117,249, LAs=318, months=369)
[AFTER FILTER 2015+] gigabit_pct: 99.4% | households_2021: 99.4% | rent_index: 90.6%


,LA_code,month,hedonic_price,n_obs,hedonic_ix,LA_name,median_price,mean_price,n_sales,is_thin,...,fastfood_per_1k_hh_z,supermkt_per_1k_hh_z,stops_per_km2_z,stops_per_10k_hh_z,imd_score_mean_z,airbnb_per_10k_hh_z,iphrp_england,iphrp_yoy,iphrp_england_z,iphrp_yoy_z
240,E06000001,2015-01-31,97716.460192,77,373.358305,Hartlepool,85000.0,117363.181818,77,False,...,0.410539,-0.489004,0.084288,0.476515,1.949175,NaN,81.608616,NaN,-1.269946,NaN
241,E06000001,2015-02-28,98614.598802,75,376.789943,Hartlepool,107200.0,123704.266667,75,False,...,0.410539,-0.489004,0.084288,0.476515,1.949175,NaN,81.608616,NaN,-1.269946,NaN
242,E06000001,2015-03-31,97404.099245,127,372.164826,Hartlepool,99995.0,122474.551181,127,False,...,0.410539,-0.489004,0.084288,0.476515,1.949175,NaN,81.870067,NaN,-1.245453,NaN
243,E06000001,2015-04-30,100311.910891,97,383.275090,Hartlepool,110000.0,114106.175258,97,False,...,0.410539,-0.489004,0.084288,0.476515,1.949175,NaN,82.212893,NaN,-1.213337,NaN
244,E06000001,2015-05-31,100404.313669,124,383.628145,Hartlepool,98497.5,131663.153226,124,False,...,0.410539,-0.489004,0.084288,0.476515,1.949175,NaN,82.586593,NaN,-1.178328,NaN


Saved merged panel (2015+) → ../data/interim/la_month_panel_2015plus.parquet (rows=40,999, LAs=318, months=129)


## Feature Engineering


In [19]:
# Helper to build rolling features for one panel
def add_rolling_features(g: pd.DataFrame) -> pd.DataFrame:
    g = g.sort_values('month').copy()

    # Reindex to continuous month-end calendar per LA (handles occasional gaps)
    full_months = pd.period_range(g['month'].min(), g['month'].max(), freq='M').to_timestamp('M')
    g = g.set_index('month').reindex(full_months)
    g.index.name = 'month'

    # Carry identifiers forward/back
    g['LA_code'] = g['LA_code'].ffill().bfill()
    if 'LA_name' in g.columns:
        g['LA_name'] = g['LA_name'].ffill().bfill()

    #  Core price series
    price = g['px_used']
    g['ret_1m'] = np.log(price).diff(1)
    g['ret_3m'] = np.log(price).diff(3)
    g['ret_6m'] = np.log(price).diff(6)
    g['mom_3m'] = g['px_used'].rolling(3, min_periods=3).mean() / g['px_used'].shift(3) - 1
    g['mom_6m'] = g['px_used'].rolling(6, min_periods=6).mean() / g['px_used'].shift(6) - 1
    g['vol_6m'] = g['ret_1m'].rolling(6, min_periods=6).std() * np.sqrt(12)
    g['n_sales_3m'] = g['n_sales'].rolling(3, min_periods=1).sum()
    g['thin_flag']  = (g['n_sales'] < 20).astype('int8')
    g['drawdown'] = g['px_used'] / g['px_used'].cummax() - 1

    #  Optional rent dynamics
    has_rent = 'rent_index' in g.columns
    if has_rent:
        g['rent_yoy'] = g['rent_index'].pct_change(12)
        g['hpi_yoy']  = g['px_used'].pct_change(12)
        g['rent_minus_price_yoy'] = g['rent_yoy'] - g['hpi_yoy']

    #  Targets (forward returns)
    g['y_ret_1m_fwd'] = np.log(price.shift(-1)) - np.log(price)
    g['y_ret_3m_fwd'] = np.log(price.shift(-3)) - np.log(price)
    g['y_up_1m']      = (g['y_ret_1m_fwd'] > 0).astype('Int8')

    #  Leakage guard: lag model features by 1 month
    feat_core = [
        'ret_1m','ret_3m','ret_6m',
        'mom_3m','mom_6m','vol_6m',
        'n_sales_3m','thin_flag','drawdown'
    ]
    feat_rent = ['rent_yoy','hpi_yoy','rent_minus_price_yoy'] if has_rent else []
    g[feat_core + feat_rent] = g[feat_core + feat_rent].shift(1)

    return g.reset_index()

def build_features_from(panel_path: Path) -> pd.DataFrame:
    panel_df = pd.read_parquet(panel_path)
    print(f"[Cell 5] Using {panel_path.name} | span {panel_df['month'].min().date()} → {panel_df['month'].max().date()} "
          f"| rows={len(panel_df):,}")
    feat_all = (panel_df
                .groupby('LA_code', group_keys=False)
                .apply(add_rolling_features)
                .reset_index(drop=True))
    return feat_all

def null_pct(df, cols):
    return df[cols].isna().mean().sort_values(ascending=False).round(3) * 100

# Build CORE (from ALL history)
feat_all_core_src = INTERIM / "la_month_panel_ALL.parquet"
feat_all_core     = build_features_from(feat_all_core_src)

core_cols = [
    'month','LA_code','LA_name','px_used','n_sales',
    'ret_1m','ret_3m','ret_6m','mom_3m','mom_6m','vol_6m',
    'n_sales_3m','thin_flag','drawdown',
    'y_ret_1m_fwd','y_ret_3m_fwd','y_up_1m',
    'gigabit_pct',  # broadband
    'fastfood_per_1k_hh','supermkt_per_1k_hh',  # POI density
    'stops_per_km2','stops_per_10k_hh',  # transport
    'imd_score_mean',  # deprivation
    'airbnb_per_10k_hh',  # short-term rentals
    # Z-scored versions
    'fastfood_per_1k_hh_z','supermkt_per_1k_hh_z',
    'stops_per_km2_z','stops_per_10k_hh_z',
    'imd_score_mean_z','airbnb_per_10k_hh_z',
    # Time context
    'iphrp_england','iphrp_yoy','iphrp_england_z','iphrp_yoy_z',
    # Rent index (needed for rent_gap calculation)
    'rent_index',
]

feat_core = feat_all_core[core_cols].copy()

print("\n[NULL % by feature — CORE]")
print(null_pct(feat_core, [c for c in feat_core.columns if c not in ['month','LA_code','LA_name']]))

out_core = INTERIM / "la_month_features_base.parquet"
feat_core.to_parquet(out_core, index=False)
print(f"Saved core rolling features → {out_core} "
      f"(rows={len(feat_core):,}, span {feat_core['month'].min().date()} → {feat_core['month'].max().date()})")

# Build WITHRENT (from 2015+ panel)
feat_all_rent_src = INTERIM / "la_month_panel_2015plus.parquet"
feat_all_rent     = build_features_from(feat_all_rent_src)

rent_cols  = ['rent_yoy','hpi_yoy','rent_minus_price_yoy']
feat_with_rent = feat_all_rent[core_cols + rent_cols].copy()

if rent_cols:
    print("\n[NULL % by feature — WITH RENT]")
    print(null_pct(feat_with_rent, rent_cols))

out_rent = INTERIM / "la_month_features_w_rent.parquet"
feat_with_rent.to_parquet(out_rent, index=False)
print(f"Saved rolling features (with rent) → {out_rent} (rows={len(feat_with_rent):,}, "
      f"span {feat_with_rent['month'].min().date()} → {feat_with_rent['month'].max().date()})")

# Quick peek
display(feat_core.head())

[Cell 5] Using la_month_panel_ALL.parquet | span 1995-01-31 → 2025-09-30 | rows=117,249

[NULL % by feature — CORE]
airbnb_per_10k_hh       86.2
airbnb_per_10k_hh_z     86.2
rent_index              68.4
iphrp_yoy_z             68.3
iphrp_yoy               68.3
iphrp_england_z         65.1
iphrp_england           65.1
imd_score_mean           9.8
imd_score_mean_z         9.8
mom_6m                   2.2
vol_6m                   2.2
ret_6m                   2.0
mom_3m                   1.3
ret_3m                   1.2
y_ret_3m_fwd             0.9
stops_per_10k_hh_z       0.7
supermkt_per_1k_hh_z     0.7
fastfood_per_1k_hh_z     0.7
supermkt_per_1k_hh       0.7
stops_per_10k_hh         0.7
fastfood_per_1k_hh       0.7
gigabit_pct              0.7
ret_1m                   0.7
y_ret_1m_fwd             0.4
drawdown                 0.3
n_sales_3m               0.3
thin_flag                0.3
stops_per_km2            0.1
n_sales                  0.1
stops_per_km2_z          0.1
px_used       

,month,LA_code,LA_name,px_used,n_sales,ret_1m,ret_3m,ret_6m,mom_3m,mom_6m,...,supermkt_per_1k_hh_z,stops_per_km2_z,stops_per_10k_hh_z,imd_score_mean_z,airbnb_per_10k_hh_z,iphrp_england,iphrp_yoy,iphrp_england_z,iphrp_yoy_z,rent_index
0,1995-01-31,E06000001,Hartlepool,26172.301189,86.0,NaN,NaN,NaN,NaN,NaN,...,-0.489004,0.084288,0.476515,1.949175,NaN,NaN,NaN,NaN,NaN,NaN
1,1995-02-28,E06000001,Hartlepool,26026.923857,86.0,NaN,NaN,NaN,NaN,NaN,...,-0.489004,0.084288,0.476515,1.949175,NaN,NaN,NaN,NaN,NaN,NaN
2,1995-03-31,E06000001,Hartlepool,26080.580175,117.0,-0.005570,NaN,NaN,NaN,NaN,...,-0.489004,0.084288,0.476515,1.949175,NaN,NaN,NaN,NaN,NaN,NaN
3,1995-04-30,E06000001,Hartlepool,26943.653925,76.0,0.002059,NaN,NaN,NaN,NaN,...,-0.489004,0.084288,0.476515,1.949175,NaN,NaN,NaN,NaN,NaN,NaN
4,1995-05-31,E06000001,Hartlepool,26342.860263,100.0,0.032557,0.029046,NaN,0.006804,NaN,...,-0.489004,0.084288,0.476515,1.949175,NaN,NaN,NaN,NaN,NaN,NaN


## Price-to-Rent and Structural Factors


In [22]:
PRICE_MIN = 1.0
RENT_MIN  = 1.0
INTERIM   = Path("../data/interim")
PRMS_PATH = INTERIM / "prms_table_2_7_la_medians.parquet"

# Config
try:
    RUN_TAG
except NameError:
    RUN_TAG = "PROP_v1"

try:
    USE_RENT_FEATURES
except NameError:
    USE_RENT_FEATURES = False   # False → CORE active; True → WITHRENT active

def _slug(s: str) -> str:
    s = re.sub(r"\s+", "_", str(s).strip())
    s = re.sub(r"[^\w\-]+", "_", s)
    return s

def _strip_mode_suffix(s: str) -> str:
    return re.sub(r"__(WITHRENT|CORE)$", "", s, flags=re.IGNORECASE)

RUN_TAG = _strip_mode_suffix(_slug(RUN_TAG))

# Inputs
SRC_CORE     = INTERIM / "la_month_features_base.parquet"     # 1995–2025
SRC_WITHRENT = INTERIM / "la_month_features_w_rent.parquet"   # 2015–2025

assert SRC_CORE.exists(),     f"Missing {SRC_CORE} (run Cell 5)"
assert SRC_WITHRENT.exists(), f"Missing {SRC_WITHRENT} (run Cell 5)"

# Helper: name normaliser + match utilities (for PRMS merge)
def norm_name(x: str) -> str:
    if pd.isna(x): return x
    s = str(x).lower()
    s = re.sub(r"\b(london\s+borough\s+of|borough\s+of|city\s+of|county\s+of)\b", " ", s)
    s = re.sub(r"\b(royal|metropolitan|borough|district|city|county|unitary\s+authority|city\s+and\s+county)\b", " ", s)
    s = re.sub(r"\bua\b", " ", s)
    s = re.sub(r"\(.*?met\s+county.*?\)", " ", s)
    s = re.sub(r"[^\w\s-]", " ", s)
    s = s.replace("-", " ")
    s = re.sub(r"\s+", " ", s).strip()
    return s

# Load PRMS (one median_rent per LA)
if not PRMS_PATH.exists():
    raise FileNotFoundError("Missing PRMS rents file. Create prms_table_2_7_la_medians.parquet first.")

prms_raw = pd.read_parquet(PRMS_PATH)
need = {"LA_name","median_rent"}
missing_cols = need - set(prms_raw.columns)
assert not missing_cols, f"PRMS file missing {missing_cols}. Has {list(prms_raw.columns)}"

prms = prms_raw[["LA_name","median_rent"]].drop_duplicates("LA_name").copy()
prms["_name_norm"] = prms["LA_name"].map(norm_name)

alias_map = {
    # Cumbria reorg
    "barrow in furness": "westmorland and furness",
    "eden": "westmorland and furness",
    "south lakeland": "westmorland and furness",
    "carlisle": "cumberland",
    "allerdale": "cumberland",
    "copeland": "cumberland",
    # BCP
    "bournemouth": "bournemouth christchurch and poole",
    "christchurch": "bournemouth christchurch and poole",
    "poole": "bournemouth christchurch and poole",
    # Common variants
    "bristol city of": "bristol",
    "kingston upon hull city of": "kingston upon hull",
    "herefordshire county of": "herefordshire",
    "city of westminster": "westminster",
}
prms["_name_norm"] = prms["_name_norm"].replace(alias_map)

drop_norms = {"tyne and wear","merseyside","greater manchester","south yorkshire","west yorkshire"}
prms = prms[~prms["_name_norm"].isin(drop_norms)].copy()

def enrich_once(df_in: pd.DataFrame, label: str) -> pd.DataFrame:
    df = df_in.copy()
    # Safety
    need_cols = {"LA_code","month","px_used"}
    miss = need_cols - set(df.columns)
    assert not miss, f"[{label}] Missing required cols: {miss}"
    # Build lookup from df (LA_code -> name) for robust merge
    la_lookup = (
        df[["LA_code","LA_name"]]
          .dropna()
          .drop_duplicates()
          .assign(_name_norm=lambda d: d["LA_name"].map(norm_name))
    )
    # Attach PRMS rent by LA_code via normalised name
    pr = prms.merge(la_lookup[["_name_norm","LA_code"]], on="_name_norm", how="left", validate="many_to_one")
    still = pr["LA_code"].isna().sum()
    if still:
        print(f"[{label}] Warning: {still} PRMS names didn’t match any panel LA; they’ll be dropped.")
    pr = pr.dropna(subset=["LA_code"])[["LA_code","median_rent"]].drop_duplicates("LA_code")

    df = df.merge(pr, on="LA_code", how="left", validate="many_to_one")

    # Valuation features
    annual_rent = (12.0 * df["median_rent"]).astype("float64")
    df["price_to_rent"] = np.where(
        pd.notna(annual_rent) & (annual_rent > 0),
        df["px_used"] / annual_rent,
        np.nan
    ).astype("float32")

    df["log_ptr"] = (
        np.log(df["px_used"].clip(lower=PRICE_MIN)) - np.log(annual_rent.clip(lower=RENT_MIN))
    ).astype("float32")

    if "rent_index" in df.columns:
        df["rent_gap"] = (
            np.log(df["px_used"].clip(lower=PRICE_MIN)) - np.log(df["rent_index"].clip(lower=RENT_MIN))
        ).astype("float32")
    else:
        df["rent_gap"] = np.nan

    # Diagnostics
    print(f"\n[{label}] Span: {df['month'].min().date()} → {df['month'].max().date()} | rows={len(df):,}")
    print(f"[{label}] Null % (PTR/log_ptr/rent_gap):")
    cols = [c for c in ["price_to_rent","log_ptr","rent_gap"] if c in df.columns]
    print(df[cols].isna().mean().round(3))

    return df

# Enrich each variant from its proper source
core_in      = pd.read_parquet(SRC_CORE)       # 1995–2025, price-only features
withrent_in  = pd.read_parquet(SRC_WITHRENT)   # 2015–2025, has rent dynamics

feat_core_enriched     = enrich_once(core_in,     "CORE (1995+)")
feat_withrent_enriched = enrich_once(withrent_in, "WITHRENT (2015+)")

# Save enriched outputs
base_name     = f"la_month_features_enriched_{RUN_TAG}"
core_name     = INTERIM / f"{base_name}__CORE.parquet"
withrent_name = INTERIM / f"{base_name}__WITHRENT.parquet"
active_name   = INTERIM / f"{base_name}.parquet"

feat_core_enriched.to_parquet(core_name, index=False)
print(f"Saved CORE     → {core_name}       (rows={len(feat_core_enriched):,})")

feat_withrent_enriched.to_parquet(withrent_name, index=False)
print(f"Saved WITHRENT → {withrent_name}  (rows={len(feat_withrent_enriched):,})")

src = withrent_name if USE_RENT_FEATURES else core_name
shutil.copyfile(src, active_name)
_act = pd.read_parquet(active_name)
print(f"→ Active ({'WITHRENT' if USE_RENT_FEATURES else 'CORE'}) saved as {active_name}")
print(f"[ACTIVE SPAN] {_act['month'].min().date()} → {_act['month'].max().date()} | rows={len(_act):,}")

[CORE (1995+)] Warning: 18 PRMS names didn’t match any panel LA; they’ll be dropped.

[CORE (1995+)] Span: 1995-01-31 → 2025-09-30 | rows=117,341
[CORE (1995+)] Null % (PTR/log_ptr/rent_gap):
price_to_rent    0.083
log_ptr          0.083
rent_gap         0.684
dtype: float64
[WITHRENT (2015+)] Warning: 18 PRMS names didn’t match any panel LA; they’ll be dropped.

[WITHRENT (2015+)] Span: 2015-01-31 → 2025-09-30 | rows=41,021
[WITHRENT (2015+)] Null % (PTR/log_ptr/rent_gap):
price_to_rent    0.082
log_ptr          0.082
rent_gap         0.095
dtype: float64
Saved CORE     → ../data/interim/la_month_features_enriched_PROP_v1__CORE.parquet       (rows=117,341)
Saved WITHRENT → ../data/interim/la_month_features_enriched_PROP_v1__WITHRENT.parquet  (rows=41,021)
→ Active (CORE) saved as ../data/interim/la_month_features_enriched_PROP_v1.parquet
[ACTIVE SPAN] 1995-01-31 → 2025-09-30 | rows=117,341


## Regime Flags


In [25]:
# Regime flags

# Load enriched features (created in Cell 14) and add regime flags
INTERIM = Path("../data/interim")

try:
    RUN_TAG
except NameError:
    RUN_TAG = "PROP_v1"

base_enriched = f"la_month_features_enriched_{RUN_TAG}"
core_path = INTERIM / f"{base_enriched}__CORE.parquet"
withrent_path = INTERIM / f"{base_enriched}__WITHRENT.parquet"

def add_regime_flags(df: pd.DataFrame) -> pd.DataFrame:
    """Add regime flags to a dataframe with 'month' column"""
    df = df.copy()
    # Ensure month is month-end timestamp (defensive)
    df["month"] = pd.to_datetime(df["month"]) + pd.offsets.MonthEnd(0)
    m = df["month"].dt.to_period("M")

    # COVID (UK housing disruption): Mar 2020 – Jun 2021
    df["covid_regime"] = m.between("2020-03", "2021-06").astype("int8")

    # BoE rate-hike regime: from Dec 2021 onward
    df["rate_hike_reg"] = (m >= "2021-12").astype("int8")

    return df

# Process CORE features
if core_path.exists():
    feat_core = pd.read_parquet(core_path)
    # Only add flags if they don't already exist
    if "covid_regime" not in feat_core.columns or "rate_hike_reg" not in feat_core.columns:
        feat_core = add_regime_flags(feat_core)
        feat_core.to_parquet(core_path, index=False)
        print(f"[CORE] Added regime flags → {core_path.name}")
    else:
        print(f"[CORE] Regime flags already present in {core_path.name}")

    # Coverage check
    for col in ["covid_regime", "rate_hike_reg"]:
        if col in feat_core.columns:
            on_rate = 100 * feat_core[col].mean()
            print(f"[REGIME CORE] {col}: {on_rate:.1f}% of rows flagged")
else:
    print(f"[WARN] CORE features not found: {core_path}")

# Process WITHRENT features
if withrent_path.exists():
    feat_withrent = pd.read_parquet(withrent_path)
    # Only add flags if they don't already exist
    if "covid_regime" not in feat_withrent.columns or "rate_hike_reg" not in feat_withrent.columns:
        feat_withrent = add_regime_flags(feat_withrent)
        feat_withrent.to_parquet(withrent_path, index=False)
        print(f"[WITHRENT] Added regime flags → {withrent_path.name}")
    else:
        print(f"[WITHRENT] Regime flags already present in {withrent_path.name}")

    # Coverage check
    for col in ["covid_regime", "rate_hike_reg"]:
        if col in feat_withrent.columns:
            on_rate = 100 * feat_withrent[col].mean()
            print(f"[REGIME WITHRENT] {col}: {on_rate:.1f}% of rows flagged")
else:
    print(f"[WARN] WITHRENT features not found: {withrent_path}")


[CORE] Added regime flags → la_month_features_enriched_PROP_v1__CORE.parquet
[REGIME CORE] covid_regime: 4.3% of rows flagged
[REGIME CORE] rate_hike_reg: 12.5% of rows flagged
[WITHRENT] Added regime flags → la_month_features_enriched_PROP_v1__WITHRENT.parquet
[REGIME WITHRENT] covid_regime: 12.4% of rows flagged
[REGIME WITHRENT] rate_hike_reg: 35.7% of rows flagged


## Forward Returns (Targets)


In [28]:
INTERIM = Path("../data/interim")
RUN_TAG = "PROP_v1"

CORE_SRC = INTERIM / "la_month_features_base.parquet"   # from Cell 5 (1995–2025)
assert CORE_SRC.exists(), f"Missing {CORE_SRC} — run Cell 5 first."

core = pd.read_parquet(CORE_SRC)

def add_targets_per_la(g: pd.DataFrame) -> pd.DataFrame:
    g = g.sort_values('month').copy()
    p = g['px_used'].astype('float64')
    g['y_ret_1m_fwd'] = np.log(p.shift(-1) / p)
    g['y_ret_3m_fwd'] = np.log(p.shift(-3) / p)
    g['y_has_1m'] = g['y_ret_1m_fwd'].notna().astype('int8')
    g['y_has_3m'] = g['y_ret_3m_fwd'].notna().astype('int8')
    g['y_up_1m']  = np.where(g['y_has_1m'] == 1, (g['y_ret_1m_fwd'] > 0).astype('int8'), pd.NA)
    g['y_up_3m']  = np.where(g['y_has_3m'] == 1, (g['y_ret_3m_fwd'] > 0).astype('int8'), pd.NA)
    return g

targets_full = (core[['LA_code','month','px_used']]
                .groupby('LA_code', group_keys=False)
                .apply(add_targets_per_la)
                .reset_index(drop=True))

# keep only compact target columns to save space
Y_COLS = ["y_ret_1m_fwd","y_ret_3m_fwd","y_up_1m","y_up_3m"]
targets_out = INTERIM / f"la_month_features_targets_{RUN_TAG}.parquet"
targets_full[['LA_code','month'] + Y_COLS].to_parquet(targets_out, index=False)

print(f"[TARGETS] span {targets_full['month'].min().date()} → {targets_full['month'].max().date()} | rows={len(targets_full):,}")
print("[TARGETS] Null %:")
print(targets_full[Y_COLS].isna().mean().round(3))
print(f"Saved targets → {targets_out}")

[TARGETS] span 1995-01-31 → 2025-09-30 | rows=117,341
[TARGETS] Null %:
y_ret_1m_fwd    0.004
y_ret_3m_fwd    0.009
y_up_1m         0.004
y_up_3m         0.009
dtype: float64
Saved targets → ../data/interim/la_month_features_targets_PROP_v1.parquet


## Select Trainable Subset and Assign Folds


In [31]:
INTERIM = Path("../data/interim")
RUN_TAG = "PROP_v1"

base_enriched = f"la_month_features_enriched_{RUN_TAG}"
paths_enriched = {
    "WITHRENT": INTERIM / f"{base_enriched}__WITHRENT.parquet",
    "CORE":     INTERIM / f"{base_enriched}__CORE.parquet",
}
targets_path = INTERIM / f"la_month_features_targets_{RUN_TAG}.parquet"

VAL_YEARS = {2020: "A_val2020", 2022: "B_val2022", 2023: "C_val2023"}
Y_COLS = ["y_ret_1m_fwd","y_ret_3m_fwd","y_up_1m","y_up_3m"]

def ensure_regimes(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    m = pd.to_datetime(df["month"]).dt.to_period("M")
    if "covid_regime" not in df.columns:
        df["covid_regime"] = m.between("2020-03", "2021-06").astype("int8")
    if "rate_hike_reg" not in df.columns:
        df["rate_hike_reg"] = (m >= "2021-12").astype("int8")
    return df

def assign_fold(dt):
    y = pd.Timestamp(dt).year
    return VAL_YEARS.get(y, "train")

def attach_targets(feat_variant: pd.DataFrame) -> pd.DataFrame:
    # Use file from Cell 8 (built on full CORE span)
    if not targets_path.exists():
        raise FileNotFoundError(f"Targets file not found: {targets_path}. Run Cell 8.")

    t = pd.read_parquet(targets_path)[["LA_code","month"] + Y_COLS]
    merged = feat_variant.merge(t, on=["LA_code","month"], how="inner", suffixes=("", "_tgt"))

    # Coalesce any accidental suffixes (future-proof)
    for col in Y_COLS:
        if col in merged.columns and f"{col}_tgt" in merged.columns:
            merged[col] = merged[col].where(merged[col].notna(), merged[f"{col}_tgt"])
            merged.drop(columns=[f"{col}_tgt"], inplace=True)
        elif f"{col}_tgt" in merged.columns:
            merged.rename(columns={f"{col}_tgt": col}, inplace=True)

    # Warn if early years were dropped (usually because the variant starts later)
    pre_min = feat_variant["month"].min()
    post_min = merged["month"].min()
    if post_min > pre_min:
        print(f"[WARN] attach_targets: rows before {post_min.date()} were dropped (no targets for those months in this variant).")

    return merged

def build_trainable_for_variant(variant: str, path: Path) -> pd.DataFrame:
    if not path.exists():
        print(f"Skipped {variant}: {path.name} not found. (Run Cell 6 to create it.)")
        return pd.DataFrame()

    print(f"\n Building {variant} from {path.name} ")
    feat_variant = pd.read_parquet(path)
    feat_variant = ensure_regimes(feat_variant)
    df = attach_targets(feat_variant)

    base_feats = [
        'ret_1m','ret_3m','ret_6m',
        'mom_3m','mom_6m','vol_6m',
        'n_sales','n_sales_3m','thin_flag','drawdown',
        'gigabit_pct','households_2021','covid_regime','rate_hike_reg'
    ]
    rent_feats = ['price_to_rent','log_ptr','rent_gap','rent_yoy','hpi_yoy','rent_minus_price_yoy']
    feature_cols = base_feats + (rent_feats if variant == "WITHRENT" else [])
    feature_cols = [c for c in feature_cols if c in df.columns]

    # Exclude columns that are all null (they would cause dropna to remove all rows)
    feature_cols = [c for c in feature_cols if df[c].notna().any()]

    target_cols = [c for c in Y_COLS if c in df.columns]
    trainable = df.dropna(subset=feature_cols + target_cols).copy()

    trainable['fold']  = trainable['month'].apply(assign_fold)
    trainable['split'] = np.where(trainable['fold'].astype(str).str.startswith(('A','B','C')), 'val', 'train')

    print(f"[TRAINABLE {variant}] shape: {trainable.shape}")
    print(f"[TRAINABLE {variant}] date span: {trainable['month'].min().date()} → {trainable['month'].max().date()}")
    print("\n[rows by split]\n", trainable['split'].value_counts().sort_index())
    print("\n[rows by fold]\n", trainable['fold'].value_counts().sort_index())

    out = INTERIM / f"la_month_trainable_{RUN_TAG}__{variant}.parquet"
    trainable.to_parquet(out, index=False)
    print(f"Saved trainable ({variant}) → {out} "
          f"(rows={len(trainable):,}, features={len(feature_cols)}, targets={len(target_cols)})\n")
    return trainable

# Build both present
trainables = {}
for variant, p in paths_enriched.items():
    trn = build_trainable_for_variant(variant, p)
    if not trn.empty:
        trainables[variant] = trn


 Building WITHRENT from la_month_features_enriched_PROP_v1__WITHRENT.parquet 
[TRAINABLE WITHRENT] shape: (32436, 47)
[TRAINABLE WITHRENT] date span: 2016-02-29 → 2025-06-30

[rows by split]
 split
train    22104
val      10332
Name: count, dtype: int64

[rows by fold]
 fold
A_val2020     3444
B_val2022     3444
C_val2023     3444
train        22104
Name: count, dtype: int64
Saved trainable (WITHRENT) → ../data/interim/la_month_trainable_PROP_v1__WITHRENT.parquet (rows=32,436, features=19, targets=4)


 Building CORE from la_month_features_enriched_PROP_v1__CORE.parquet 
[TRAINABLE CORE] shape: (113112, 44)
[TRAINABLE CORE] date span: 1995-08-31 → 2025-06-30

[rows by split]
 split
train    101772
val       11340
Name: count, dtype: int64

[rows by fold]
 fold
A_val2020      3780
B_val2022      3780
C_val2023      3780
train        101772
Name: count, dtype: int64
Saved trainable (CORE) → ../data/interim/la_month_trainable_PROP_v1__CORE.parquet (rows=113,112, features=13, targets=4)



## Save Outputs


In [34]:
INTERIM = Path("../data/interim")

try:
    RUN_TAG
except NameError:
    RUN_TAG = "PROP_v1"

try:
    USE_RENT_FEATURES
except NameError:
    USE_RENT_FEATURES = False  # False → CORE active; True → WITHRENT active

base_enriched = f"la_month_features_enriched_{RUN_TAG}"
feat_paths = {
    "WITHRENT": INTERIM / f"{base_enriched}__WITHRENT.parquet",
    "CORE":     INTERIM / f"{base_enriched}__CORE.parquet",
}

train_paths = {
    "WITHRENT": INTERIM / f"la_month_trainable_{RUN_TAG}__WITHRENT.parquet",
    "CORE":     INTERIM / f"la_month_trainable_{RUN_TAG}__CORE.parquet",
}

active_variant = "WITHRENT" if USE_RENT_FEATURES else "CORE"

# Safety checks
for v in ["WITHRENT", "CORE"]:
    if not feat_paths[v].exists():
        print(f"[WARN] Missing features file for {v}: {feat_paths[v]}")
    if not train_paths[v].exists():
        print(f"[WARN] Missing trainable file for {v}: {train_paths[v]}")

# Build “active” aliases (copy)
active_feat_src   = feat_paths[active_variant]
active_train_src  = train_paths[active_variant]

active_feat_dst   = INTERIM / f"la_month_features_ACTIVE_{RUN_TAG}.parquet"
active_train_dst  = INTERIM / f"la_month_trainable_ACTIVE_{RUN_TAG}.parquet"

shutil.copyfile(active_feat_src, active_feat_dst)
shutil.copyfile(active_train_src, active_train_dst)

print(f"→ Active features  ({active_variant}) → {active_feat_dst}")
print(f"→ Active trainable ({active_variant}) → {active_train_dst}")

# Summaries
manifest = {"RUN_TAG": RUN_TAG, "active_variant": active_variant, "variants": {}}

def _summarise(path: Path):
    if not path.exists():
        return None
    df = pd.read_parquet(path)
    info = {
        "rows": int(len(df)),
        "cols": list(df.columns),
        "span": {
            "min_month": pd.to_datetime(df["month"]).min().date().isoformat(),
            "max_month": pd.to_datetime(df["month"]).max().date().isoformat(),
        }
    }
    return info

for v in ["WITHRENT", "CORE"]:
    manifest["variants"][v] = {
        "features_path": str(feat_paths[v]),
        "trainable_path": str(train_paths[v]),
        "features_summary": _summarise(feat_paths[v]),
        "trainable_summary": _summarise(train_paths[v]),
    }

manifest_path = INTERIM / f"la_month_manifest_{RUN_TAG}.json"
with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2)

print(f"Wrote manifest → {manifest_path}")

# Printout
act_feat = pd.read_parquet(active_feat_dst)
act_trn  = pd.read_parquet(active_train_dst)
print(f"[ACTIVE] Features span: {act_feat['month'].min().date()} → {act_feat['month'].max().date()} | rows={len(act_feat):,}")
print(f"[ACTIVE] Trainable span: {act_trn['month'].min().date()} → {act_trn['month'].max().date()} | rows={len(act_trn):,}")
print(f"[ACTIVE] Feature cols (first 10): {act_trn.columns[:10].tolist()} …")

→ Active features  (CORE) → ../data/interim/la_month_features_ACTIVE_PROP_v1.parquet
→ Active trainable (CORE) → ../data/interim/la_month_trainable_ACTIVE_PROP_v1.parquet
Wrote manifest → ../data/interim/la_month_manifest_PROP_v1.json
[ACTIVE] Features span: 1995-01-31 → 2025-09-30 | rows=117,341
[ACTIVE] Trainable span: 1995-08-31 → 2025-06-30 | rows=113,112
[ACTIVE] Feature cols (first 10): ['month', 'LA_code', 'LA_name', 'px_used', 'n_sales', 'ret_1m', 'ret_3m', 'ret_6m', 'mom_3m', 'mom_6m'] …
